# Analysis of HPC Sampling Results

This notebook loads and analyzes the MCMC chains generated by the HPC jobs on COSMA7.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in path
sys.path.insert(0, os.path.abspath('..'))

from src.cmb import load_cmb_chains
from src.cmb.power import call_CAMB_map

## 1. Load Chains
Update the `results_dir` to point to the output folder defined in your SLURM script.

In [ ]:
LMAX = 200
NSIDE = 128
results_dir = f'../results/lmax{LMAX}_nside{NSIDE}_nuts'

if os.path.exists(results_dir):
    chains_samples, chains_logprob, chains_accepted = load_cmb_chains(results_dir)
    print(f"Mean acceptance rate: {np.nanmean(chains_accepted):.3f}")
else:
    print(f"Results directory not found: {results_dir}")

## 2. Convergence Diagnostics
Check the log-posterior traces and Gelman-Rubin R-hat.

In [ ]:
# Log-posterior traces
plt.figure(figsize=(12, 4))
for i, lp in enumerate(chains_logprob):
    plt.plot(lp, label=f'Chain {i+1}', alpha=0.7)
plt.xlabel('Sample')
plt.ylabel('Log-posterior')
plt.title('HMC/NUTS Trace')
plt.legend()
plt.show()

## 3. Power Spectrum Inference
Compare the inferred spectrum with the fiducial ΛCDM model.

In [ ]:
# Pool samples
all_samples = np.concatenate(chains_samples, axis=0)
n_lncl = LMAX - 2
ln_cl_samples = all_samples[:, :n_lncl]
cl_samples = np.exp(ln_cl_samples)

ells = np.arange(2, LMAX)
cl_mean = cl_samples.mean(axis=0)
dl_mean = ells * (ells + 1) * cl_mean / (2 * np.pi)

# Fiducial LCDM
LCDM_PARAMS = [67.74, 0.0486, 0.2589, 0.06, 0.0, 0.066]
cl_lcdm = call_CAMB_map(LCDM_PARAMS, LMAX)
dl_lcdm = ells * (ells + 1) * cl_lcdm[2:LMAX] / (2 * np.pi)

plt.figure(figsize=(10, 6))
plt.plot(ells, dl_lcdm, 'r-', label='$\Lambda (fiducial)')
plt.plot(ells, dl_mean, 'k-', label='Inferred (HPC mean)')
plt.xlabel('ell')
plt.ylabel('D_ell')
plt.legend()
plt.show()